# Reggie Reports
Put location of data and shadowbook csv files below.

In [1]:
shadow=r'/kaggle/input/accounts/access.csv' 
data=r'/kaggle/input/accounts/data.csv'

#### Change nothing below:

In [2]:
from datetime import datetime
import time
start=time.time()
date=datetime.today().strftime('%m-%d-%Y')
print("Running Reggie Report on: ", date)

Running Reggie Report on:  08-12-2022


# ShadowBook
Shadowbook be in the form of Access, SQL, Excel, or another software that can export to csv. 

In [3]:
import pandas as pd

aDF=pd.read_csv(shadow)
aDF.head()

,VOUCHER#,VOUCHER DATE,PROCESS DATE,INVOICE#,INVOICE DATE,VENDOR,AMOUNT,DESCRIPTION,VOU#,REQ#,ACCOUNT ID,FY
0,V12345,8/8/2022,8/8/2022,JE123456,8/8/2022,WALMART,387.52,Food,PR,Y,20504-05-987000402-123456,FY23
1,V12346,8/9/2022,8/8/2022,JE123457,8/9/2022,TESLA,962.14,Car,PR,Y,20504-05-987000402-987654,FY23
2,V12347,8/10/2022,8/8/2022,JE123458,8/10/2022,AMAZON,17.20,Stuff,PR,Y,20504-05-987000402-123456,FY23
3,V12348,8/11/2022,8/8/2022,JE123459,8/11/2022,AMAZON,26.38,Computer,PR,Y,20504-05-987000402-888999,FY23
4,V12349,8/12/2022,8/8/2022,JE123460,8/12/2022,TACO,45.00,NaN,PR,Y,98204-05-987000402-777666,FY23


#### Scan for Duplicate Entries

In [4]:
def findDups():
    dups=''
    dupRows = aDF[aDF.duplicated(['VOUCHER#', 'AMOUNT','ACCOUNT ID', 'INVOICE#'])]
    return list(dupRows['VOUCHER#'])    
findDups=findDups()

#### Scan for Missing Values

In [5]:
def findMissing():
    missingDF=aDF[['VOUCHER#','AMOUNT', 'ACCOUNT ID', 'INVOICE#']].isnull()
    missingDF.head()
findMissing()

#### Filter Important Columns

In [6]:
aDF=aDF[['AMOUNT', 'ACCOUNT ID', 'REQ#']]
aDF.head()

,AMOUNT,ACCOUNT ID,REQ#
0,387.52,20504-05-987000402-123456,Y
1,962.14,20504-05-987000402-987654,Y
2,17.20,20504-05-987000402-123456,Y
3,26.38,20504-05-987000402-888999,Y
4,45.00,98204-05-987000402-777666,Y


#### Remove Hypens

In [7]:
aDF['ACCOUNT ID'] = aDF['ACCOUNT ID'].astype(str).apply(lambda x: x.replace('-', ''))
aDF.head()

,AMOUNT,ACCOUNT ID,REQ#
0,387.52,2050405987000402123456,Y
1,962.14,2050405987000402987654,Y
2,17.20,2050405987000402123456,Y
3,26.38,2050405987000402888999,Y
4,45.00,9820405987000402777666,Y


#### Find only POSTED access records

In [8]:
aDF=aDF[aDF['REQ#'].str.contains('Y')]
aDF.head()

,AMOUNT,ACCOUNT ID,REQ#
0,387.52,2050405987000402123456,Y
1,962.14,2050405987000402987654,Y
2,17.20,2050405987000402123456,Y
3,26.38,2050405987000402888999,Y
4,45.00,9820405987000402777666,Y


#### Add each amount by each accounting total

In [9]:
aDF=aDF.groupby('ACCOUNT ID')['AMOUNT'].sum().reset_index().round(2)
aDF.head()

,ACCOUNT ID,AMOUNT
0,2050405987000402123456,2293.95
1,2050405987000402888999,26.38
2,2050405987000402987654,962.14
3,9820405987000402444333,265.50
4,9820405987000402777666,45.00


# Data Transactions

In [10]:
df=pd.read_csv(data)
df.head()

,GL Account,GL Description,Date,Source,Reference Number,Description,Opening Balance,YTD Debits,YTD Credits
0,20504-05-987000402-123456,ReggieCR,8/8/2022,JE,AAVCDF6100,PR,NaN,387.52,NaN
1,20504-05-987000402-987654,ReggieCR,8/8/2022,JE,AAVCDF6101,PR,NaN,962.14,NaN
2,20504-05-987000402-123456,ReggieCR,8/8/2022,JE,AAVCDF6102,PR,NaN,17.20,NaN
3,20504-05-987000402-888999,ReggieCR,8/8/2022,JE,AAVCDF6103,PR,NaN,26.38,NaN
4,98204-05-987000402-777666,ReggieCR,8/8/2022,JE,AAVCDF6104,PR,NaN,45.00,NaN


#### Fill empty YTD Credit with 0

In [11]:
df[' YTD Credits']=df[' YTD Credits'].fillna(0)
df.head()

,GL Account,GL Description,Date,Source,Reference Number,Description,Opening Balance,YTD Debits,YTD Credits
0,20504-05-987000402-123456,ReggieCR,8/8/2022,JE,AAVCDF6100,PR,NaN,387.52,0.0
1,20504-05-987000402-987654,ReggieCR,8/8/2022,JE,AAVCDF6101,PR,NaN,962.14,0.0
2,20504-05-987000402-123456,ReggieCR,8/8/2022,JE,AAVCDF6102,PR,NaN,17.20,0.0
3,20504-05-987000402-888999,ReggieCR,8/8/2022,JE,AAVCDF6103,PR,NaN,26.38,0.0
4,98204-05-987000402-777666,ReggieCR,8/8/2022,JE,AAVCDF6104,PR,NaN,45.00,0.0


#### Debits minus credits

In [12]:
df = df.assign(DataTotal = df[' YTD Debits'] - df[' YTD Credits'])
df.head()                                                   

,GL Account,GL Description,Date,Source,Reference Number,Description,Opening Balance,YTD Debits,YTD Credits,DataTotal
0,20504-05-987000402-123456,ReggieCR,8/8/2022,JE,AAVCDF6100,PR,NaN,387.52,0.0,387.52
1,20504-05-987000402-987654,ReggieCR,8/8/2022,JE,AAVCDF6101,PR,NaN,962.14,0.0,962.14
2,20504-05-987000402-123456,ReggieCR,8/8/2022,JE,AAVCDF6102,PR,NaN,17.20,0.0,17.20
3,20504-05-987000402-888999,ReggieCR,8/8/2022,JE,AAVCDF6103,PR,NaN,26.38,0.0,26.38
4,98204-05-987000402-777666,ReggieCR,8/8/2022,JE,AAVCDF6104,PR,NaN,45.00,0.0,45.00


#### Filter by Account Numbers and Amounts

In [13]:
df=df[['GL Account', 'DataTotal']]
df.head()

,GL Account,DataTotal
0,20504-05-987000402-123456,387.52
1,20504-05-987000402-987654,962.14
2,20504-05-987000402-123456,17.20
3,20504-05-987000402-888999,26.38
4,98204-05-987000402-777666,45.00


#### Remove Hypens (temporarily)

In [14]:
df['GL Account'] = df['GL Account'].astype(str).apply(lambda x: x.replace('-', ''))
df.head()

,GL Account,DataTotal
0,2050405987000402123456,387.52
1,2050405987000402987654,962.14
2,2050405987000402123456,17.20
3,2050405987000402888999,26.38
4,9820405987000402777666,45.00


In [15]:
df = df.groupby('GL Account')['DataTotal'].sum().reset_index().round(2)
df.head()

,GL Account,DataTotal
0,1234567987654000111333,52.35
1,1234567987654000555555,56.42
2,2050405987000402123456,2946.18
3,2050405987000402888999,26.38
4,2050405987000402987654,962.14


#### Check for new Shadowbook account number
If there is an account posting in Data but then shadowbook that transaction needs to be looked into. 

In [16]:
def checkDataAcc():
    dataAccounts=list(df['GL Account'])
    shadowAccounts=list(aDF['ACCOUNT ID'])
    dataOnly=''
    
    for d in dataAccounts:
        if d not in shadowAccounts:
            dataOnly=dataOnly+d+"  "
    if len(dataOnly)>0:
        print(dataOnly)
        return "Data Accounts only (not in shadowbook): " + dataOnly
    else:
        return "No New Data Accounts"
dataOnly=checkDataAcc()

1234567987654000111333  1234567987654000555555  


#### Check for unmatched Shadowbook account
If there is an account in the Shadowbook that is not actually in Data then further looking into needs to be done.

In [17]:
def checkShadowAcc():
    dataAccounts=list(df['GL Account'])
    shadowAccounts=list(aDF['ACCOUNT ID'])
    shadowOnly=''

    for s in shadowAccounts:
        if s not in dataAccounts:
            shadowOnly=shadowOnly+s+"  "
    if len(shadowOnly)>0:
        print(shadowOnly)
        return "Check Shadow Account: " + shadowOnly
    else:
        return "No New Data Accounts"
shadowOnly=checkShadowAcc()

# Join Data and Shadowbook

#### Rename Shadowbook Account ID to GL Account to do a join

In [18]:
aDF=aDF.rename(columns={"ACCOUNT ID": "GL Account"})
df.head()

,GL Account,DataTotal
0,1234567987654000111333,52.35
1,1234567987654000555555,56.42
2,2050405987000402123456,2946.18
3,2050405987000402888999,26.38
4,2050405987000402987654,962.14


#### Join Data by GL Account

In [19]:
finalDF=pd.merge(df, aDF, on='GL Account')
finalDF=finalDF.rename(columns={"AMOUNT":"AccountingAmount"})
finalDF.head()

,GL Account,DataTotal,AccountingAmount
0,2050405987000402123456,2946.18,2293.95
1,2050405987000402888999,26.38,26.38
2,2050405987000402987654,962.14,962.14
3,9820405987000402444333,265.50,265.50
4,9820405987000402777666,45.00,45.00


#### Add Hypens again
1. Easier to copy paste account numbers later 
2. New account numbers will not turn into scientific notation if turned back into an excel sheet

In [20]:
accHyphens=[]
for a in finalDF['GL Account']:
    accHyphens.append(a[0:5]+'-'+a[6:15]+'-'+a[16:22])
    
name = df.columns[0] # Find the name of the column by index
finalDF.drop(name, axis = 1, inplace = True) #drop column
finalDF.insert(0, name, accHyphens ) #readd column that has hyphens
finalDF.head()

,GL Account,DataTotal,AccountingAmount
0,20504-598700040-123456,2946.18,2293.95
1,20504-598700040-888999,26.38,26.38
2,20504-598700040-987654,962.14,962.14
3,98204-598700040-444333,265.50,265.50
4,98204-598700040-777666,45.00,45.00


#### Calculate Difference and then sort by priority 

In [21]:
finalDF = finalDF.assign(Difference = finalDF['DataTotal'] - finalDF['AccountingAmount']).sort_values(by=['Difference'], ascending=False).reset_index(drop=True)
finalDF.head()

,GL Account,DataTotal,AccountingAmount,Difference
0,20504-598700040-123456,2946.18,2293.95,652.23
1,98204-598700040-990099,2215.04,2210.64,4.40
2,20504-598700040-888999,26.38,26.38,0.00
3,20504-598700040-987654,962.14,962.14,0.00
4,98204-598700040-444333,265.50,265.50,0.00


#### Create Excel Sheet

In [22]:
finalDF.to_csv("ReggieReport.csv", index=False)

# Create Excel sheet with Header

In [23]:
'''
#!pip install xlsxwriter
import xlsxwriter

dept = "ISU Campus Department"
intro = "Reggie Report ran on: " + date
dataAccOnly  = dataOnly
dups='No Duplicate Records Detected'

if len(findDups)>0:
    dups='Duplicate Records Detected: '
    for dup in findDups:
        dups+=dup+'  '

newFile='ReggieReport.xlsx'
writer = pd.ExcelWriter(newFile, engine='xlsxwriter')

# Write each dataframe to a different worksheet.
finalDF.to_excel(writer, sheet_name=date, startrow=5, startcol=0, header=True, index=False)

worksheet1 = writer.sheets[date]

#Write to each worksheet:
worksheet1.write(0, 0, dept)
worksheet1.write(1, 0, intro)
worksheet1.write(2, 0, dataAccOnly)
worksheet1.write(3, 0, dups)

# Close the Pandas Excel writer and output the Excel file.
writer.save()
print("New Excel report created: ", newFile)
'''

'\n#!pip install xlsxwriter\nimport xlsxwriter\n\ndept = "ISU Campus Department"\nintro = "Reggie Report ran on: " + date\ndataAccOnly  = dataOnly\ndups=\'No Duplicate Records Detected\'\n\nif len(findDups)>0:\n    dups=\'Duplicate Records Detected: \'\n    for dup in findDups:\n        dups+=dup+\'  \'\n\nnewFile=\'ReggieReport.xlsx\'\nwriter = pd.ExcelWriter(newFile, engine=\'xlsxwriter\')\n\n# Write each dataframe to a different worksheet.\nfinalDF.to_excel(writer, sheet_name=date, startrow=5, startcol=0, header=True, index=False)\n\nworksheet1 = writer.sheets[date]\n\n#Write to each worksheet:\nworksheet1.write(0, 0, dept)\nworksheet1.write(1, 0, intro)\nworksheet1.write(2, 0, dataAccOnly)\nworksheet1.write(3, 0, dups)\n\n# Close the Pandas Excel writer and output the Excel file.\nwriter.save()\nprint("New Excel report created: ", newFile)\n'

In [24]:
print("Done. Finished in ", round((time.time()-start), 4), " seconds")

Done. Finished in  1.3558  seconds
